# FLEXPART exercise — Does aerosol shape matter?

We compare FLEXPART simulations in which the **equivalent particle diameter, density, release and meteorology are kept constant**, while particle morphology is changed. This is very relevant for particles of same material but different shapes and sizes, such as microplastics!

> **If we have a large variability in shapes, which particle morphology remains airborne longer, and how does shape affect transport and deposition?**

Try these cases:

| Species | Morphology |
|---|---|
| 20 | 30 µm sphere |
| 21 | fibre, aspect ratio 10, averaged orientation |
| 22 | fibre, aspect ratio 50, averaged orientation |
| 23 | fibre, aspect ratio 50, horizontal orientation |
| 33 | flat film, averaged orientation |

Before looking at the results, write down your expected ranking from **shortest** to **longest atmospheric residence time**.

Compare total atmospheric burden with time, dry and wet deposition, and spatial transport patterns.

### We need to define the SPECIES:

*Beware that, while for the cylinders the dimensions from the equivalent spherical diameter are automatically computed, for the flat film you will have to compute the three sides of the film. **We will suppose that the film will have 1 um thickness, and that it is a square**!*

Volume of the film=PLA x PIA x PSA with PLA=PIA and PSA=1 micron


- Let’s say that we want to simulate a synthetic fiber, in polyester, that we said had density of around 1400 kg/m3

PDENSITY=1.4E3,              ! Dry deposition (particles) – rho

- Let’s have a size distribution with standard deviation of 1.25 around the main equivalent diameter

PDSIGMA=1.25

- Finally for the scavenging properties…let’s say it is very easily scavenged below cloud, it is a good cloud nuclei but a poor ice nuclei (for microplastic it is not well understood so far, hence it is only just an hypothesis)

 PCRAIN_AERO=1.0,             ! Below-cloud scavenging (particles) - Crain (crain_aero) [arbitrary value]
 
 PCSNOW_AERO=1.0,             ! Below-cloud scavenging (particles) - Csnow (csnow_aero) [arbitrary value]
 
 PCCN_AERO=0.9,               ! In-cloud scavenging (particles) - CCNeff (ccn_aero) [arbitrary value]
 
 PIN_AERO=0.1,                ! In-cloud scavenging (particles) - INeff (in_aero) [arbitrary value]

### Let's test it with the following simulation with COMMAND, OUTGRID and RELEASE:

- We make a forward simulation, lasting 7 days, from 2017-09-25 00:00

That means, start and end times are between 2017-09-25 00:00 and 2017-10-01 00:00

- Output time step to 1 hour

- Gridded **output** between 20-65 E and 30-55 N with 0.5 space resolution, and from ground to 2500 m, with 100 m vertical resolution. I strongly suggest to use netcdf output (IOUT=9)

- **Release** at 35 E and 40 N, between 0-1000m , with 5000 particles, 1 kg of mass (MASS=1)

BEWARE: if you see that the script crashes in the binder after some time, lower the number of particles. e.g. 2000.

- We also want to have some turbulence here:
CTL = 5,  IFINE =10, LTURBULENCE=1

- And we take into account convection too:
LCONVECTION=1

- We released in mass we want the concentration in mass too. Deposition as well, in forward mode. Hence… 


IND_SOURCE=            1, ! Unit to be used at the source; [1]mass 2]mass mixing ratio
IND_RECEPTOR=          1, ! Unit to be used at the receptor; [1]mass 2]mass mixing ratio 3]wet depo. 4]dry depo.

In [4]:
## Adviced pyhon libraries

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature


## 1. Find and load the FLEXPART output

Let's rename the output data of each species as 'grid_conc_YYYYMMDDHHMMSS_xx.nc'., with the xx indicating the number of the species. For example 020:  "grid_conc_YYYYMMDDHHMMSS_20.nc"


### What do we need to use?

- `spec001_mr`: atmospheric mass concentration
- `DD_spec001`: accumulated dry deposition
- `WD_spec001`: accumulated wet deposition
- `longitude`, `latitude`, `height`, `time`: coordinates



## 2. Integrate the gridded output

Let's estimate the total

- **airborne mass** by integrating concentration over the 3-D grid with time dependece;

Airborne mass gives at each time step the concentration of the suspended aerosol

- **dry-deposited mass** over the horizontal grid with time dependece;

DD gives at each time step the dry depositon accumulated until that specific time step

- **wet-deposited mass** over the horizontal grid with time dependece.

Similarly, WD gives at each time step the wet depositon accumulated until that specific time step


### Here are some useful scripts, assuming that you will open data with

files = { \
    "Sphere": PutHereTheRightFile, \
    "Fibre AR=10": PutHereTheRightFile, \
    "Fibre AR=50": PutHereTheRightFile, \
    "Fibre AR=50 horizontal": PutHereTheRightFile, \
    "Flat film": PutHereTheRightFile" \
}

datasets = {} \

for label, filename in files.items(): datasets[label] = xr.open_dataset(filename) 

In [5]:
plt.rcParams["figure.figsize"] = (9, 5) #Let's choose here the general sizes of our plots
EARTH_RADIUS = 6_371_000.0 # in meters 

#With this we compute the area of grid
def gridcell_areas(ds):
    lon = ds.longitude.values
    lat = ds.latitude.values
    dlon_deg = abs(np.median(np.diff(lon)))
    dlat_deg = abs(np.median(np.diff(lat)))
    dlon = np.deg2rad(dlon_deg)
    lat1 = np.deg2rad(lat - dlat_deg / 2)
    lat2 = np.deg2rad(lat + dlat_deg / 2)
    area_lat = EARTH_RADIUS**2 * dlon * (np.sin(lat2) - np.sin(lat1))
    area = np.repeat(area_lat[:, None], len(lon), axis=1)
    return xr.DataArray(area,
                        coords={"latitude": ds.latitude, "longitude": ds.longitude},
                        dims=("latitude", "longitude"))

#With this we want to compute the thickness of the considered layer
def layer_thickness(ds):
    # FLEXPART gives us the TOP of each vertical output layer
    # The bottom of the first layer is the ground (0 m), for all other layers, the bottom is the top of the previous layer.
    layer_top = ds.height.values
    layer_bottom = np.concatenate(([0], layer_top[:-1]))
    thickness = layer_top - layer_bottom
    # Put the same vertical coordinate as the FLEXPART data
    thickness = xr.DataArray(
        thickness,
        coords={"height": ds.height},
        dims=["height"]
    )
    return thickness

## The computations will look like this!

- We make the conversion ng m-3 * m3 -> ng -> kg and we integrate over all layers:

airborne = (ds["spec001_mr"] * area * dz).sum(("height", "latitude", "longitude")) * 1e-12  

- Same for deposition, we do 1e-12 kg m-2 * m2 -> kg

dry = (ds["DD_spec001"] * area).sum(("latitude", "longitude")) * 1e-12

wet = (ds["WD_spec001"] * area).sum(("latitude", "longitude")) * 1e-12

## 3. Atmospheric burden through time

Let's plot the timeseries of the total airborne mass, for each morphology. 
This will tell you which of those shapes is having, with the same mass, **the longest time of suspension**

## 4. Deposition fluxes through time

We want to check the same exact thing also in dry and wet deposition. 

-**How do you expect these deposition fluxes to behave with respect to the atmospheric burden?**

## 5. Maps of airborne material (optional)

Let's try also to get a comparison of the spatial distribution of these variables (spatial map)

## 6. Comparison table at the final step
Let's take the last timestep of the simulation, integrating both in lat-lon and vertically, to obtain a table such as:

![Table](images/Table_ex.png)

## 7. Vertical distribution of airborne material

So far we have looked at the horizontal distribution of the particles.
But does particle morphology also affect their **vertical distribution**?

We can calculate a latitude–height cross section of atmospheric concentration, averaged over all longitudes, at the last 5 time steps, to see how it behaves (by last 5 time steps I mean from 2017-09-30 20:00 until 2017-10-01 00:00)

# Questions for reflection

1. Which morphology appeared to remain airborne longest? Was your prediction correct?
2. Does changing **orientation** have an effect comparable to changing aspect ratio?
3. How does the **flat film** compare with the fibres and sphere, despite equal equivalent volume and density?
4. Why can wet deposition differ even though wet-scavenging parameters are identical?
5. If you look at the table produced in table 6, you'll notice that the total sum of airborne+deposited is <1. Can you guess why?
